In [1]:
# Q6C(a-b): Set up the Tic-Tac-Toe environment and game rules.
from collections import defaultdict
import numpy as np
WIN_LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),
             (0, 3, 6), (1, 4, 7), (2, 5, 8),
             (0, 4, 8), (2, 4, 6)]
def winner(board):
    for a, b, c in WIN_LINES:
        if board[a] != 0 and board[a] == board[b] == board[c]:
            return board[a]
    return 0
def available(board):
    return [i for i, value in enumerate(board) if value == 0]
def show(board):
    symbols = {1: "X", -1: "O", 0: "."}
    print("\n".join(" ".join(symbols[v] for v in board[i:i + 3])
                    for i in (0, 3, 6)))


In [2]:
# Q6C(c): Q-learning agent X plays against a random opponent O.
# The state is the board immediately before X's turn.
q = defaultdict(lambda: np.zeros(9))
rng = np.random.default_rng(42)
alpha, gamma = 0.25, 0.95
def choose_action(board, epsilon):
    legal = available(board)
    if rng.random() < epsilon:
        return int(rng.choice(legal))
    values = q[tuple(board)][legal]
    best = [a for a, value in zip(legal, values) if value == max(values)]
    return int(rng.choice(best))
def play_episode(epsilon, learn=True):
    board = [0] * 9
    while True:
        state = tuple(board)
        action = choose_action(board, epsilon)
        board[action] = 1
        if winner(board) == 1:
            reward, result = 1.0, "win"
            break
        if not available(board):
            reward, result = 0.2, "draw"
            break
        board[int(rng.choice(available(board)))] = -1
        if winner(board) == -1:
            reward, result = -1.0, "loss"
            break
        if not available(board):
            reward, result = 0.2, "draw"
            break
        if learn:
            next_state = tuple(board)
            target = gamma * max(q[next_state][available(board)])
            q[state][action] += alpha * (target - q[state][action])
    if learn:
        q[state][action] += alpha * (reward - q[state][action])
    return result, board


In [3]:
# Q6C(d): Train through repeated games, gradually reducing exploration.
training_results = {"win": 0, "draw": 0, "loss": 0}
for episode in range(30000):
    epsilon = max(0.05, 1 - episode / 24000)
    result, _ = play_episode(epsilon)
    training_results[result] += 1
print("Training results:", training_results)
print("Learned board states:", len(q))


Training results: {'win': 25201, 'draw': 1774, 'loss': 3025}
Learned board states: 2415


In [4]:
# Q6C(e): Test the learned greedy policy against new random games.
test_results = {"win": 0, "draw": 0, "loss": 0}
for _ in range(1000):
    result, _ = play_episode(epsilon=0, learn=False)
    test_results[result] += 1
print("Test results (1000 games):", test_results)
print("Win rate:", round(test_results["win"] / 1000, 3))
print("Example game (X=agent, O=random opponent):")
result, final_board = play_episode(epsilon=0, learn=False)
show(final_board)
print("Result:", result)


Test results (1000 games): {'win': 989, 'draw': 11, 'loss': 0}
Win rate: 0.989
Example game (X=agent, O=random opponent):
O . X
. X .
X O .
Result: win
